In [ ]:
import torch
T, L = 64, 64
dev='cuda:0'

h=5
id_block = 0
pos_root = 32
pos_start = pos_root + id_block * L
pos_end = pos_start + L

order = torch.load(f'../stats_inf/0/unmask_{pos_start}_{pos_end}.pt').squeeze(-1) - pos_start
attn = torch.load(f'../stats_inf/0/attn_{pos_start}_{pos_end}.pt').squeeze(2)[:,0,:]

step_of = torch.full((L,), -1, dtype=torch.long, device=dev)
step_of[order] = torch.arange(T, device=dev)
future_gap = step_of.view(1,-1) - torch.arange(T, device=dev).view(-1,1)
cand_mask = future_gap > 0
neg_inf = torch.finfo(torch.bfloat16).min
S = attn.masked_fill(~cand_mask, neg_inf)
n_cand = (T - 1) - torch.arange(T, device=dev)                 # [T]  number of still-masked tokens after step t

In [16]:
rel = ((future_gap >= 1) & (future_gap <= h))[0]
toph = S[0].topk(h).indices
print(rel.shape, toph.shape)
hit = rel.gather(-1, toph)
print(hit)

torch.Size([64]) torch.Size([5])
tensor([ True,  True, False, False, False], device='cuda:0')


In [ ]:

idx_sorted = self.S.argsort(dim=-1, descending=True)        # [T, L]  non-candidates (-inf) sink to the end
gap_sorted = self.future_gap.gather(-1, idx_sorted)         # [T, L]

is_cand = gap_sorted > 0                                    # [T, L]  a retrieved candidate at this rank
is_pos = (gap_sorted >= 1) & (gap_sorted <= h)              # [T, L]  a soonest-h positive

tp = is_pos.cumsum(-1).double()                            # [T, L]
retrieved = is_cand.cumsum(-1).double().clamp_min(1.0)     # [T, L]  rank among candidates only
precision = tp / retrieved                                # [T, L]

P = is_pos.double().sum(-1)                                # [T]  == min(h, n_cand)
ap = (precision * is_pos.double()).sum(-1) / P.clamp_min(1.0)   # [T]

valid = self.n_cand > h                                    # need >=1 positive and >=1 negative

grade: tensor([0., 5., 4., 3., 2., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0',
       dtype=torch.float64)
pred: tensor([ 1,  2,  8, 13, 12], device='cuda:0')
dcg[0]: tensor(7.5237, device='cuda:0', dtype=torch.float64)
tensor([5., 4., 3., 2., 1.], device='cuda:0', dtype=torch.float64)
tensor([0.7325, 0.8582, 0.9623, 0.6374, 0.8785, 0.8959, 0.8073, 0.5018, 0.7029,
        0.8073, 0.7325, 0.7325, 0.5244, 0.5018, 0.6830, 0.7556, 0.8679, 0.7002,
        0.8107, 0.0753, 0.3674, 0.5706, 0.0000, 0.0000, 0.8692, 0.4930, 0.7342,
        0.4868, 0.1883, 0.8673, 0.8710, 0.4868, 0.0974, 0.8095, 0.5722, 0.0000,
        0.8337, 0.7841, 0.0000, 0.5706, 0.1325, 0.5244, 0.5992, 0.7342, 0.7197,
        0.6156, 0.5638, 0.1350, 1.0000, 0.4868, 0.3824, 0.6374, 0.3071, 0.4733,

In [ ]:
def ndcg_at_h(self, H, gain="linear"):   # graded by soonness, attention ranks the candidates; per-step [T], use .nanmean()
    dev = self.attn.device

    base = (H - (self.future_gap - 1)).clamp(min=0).double()    # [T, L]  gap=1 -> H, gap=H -> 1, gap>H -> 0
    if gain == "exp":
        base = torch.exp2(base) - 1.0
    # end
    grade = base * self.cand_mask.double()                     # [T, L]  zero out non-candidates

    disc = 1.0 / torch.log2(torch.arange(H, device=dev).double() + 2.0)   # [H]
    pred = self.S.topk(H, dim=-1).indices                      # [T, H]  predicted ranking
    dcg = (grade.gather(-1, pred) * disc).sum(-1)              # [T]

    ideal = grade.topk(H, dim=-1).values                       # [T, H]  ideal ranking
    idcg = (ideal * disc).sum(-1)                              # [T]

    ndcg = dcg / idcg.clamp_min(torch.finfo(torch.float64).tiny)   # [T]
    valid = (self.n_cand >= 2) & (idcg > 0)
    return self._nan_in
# end


def recall_at_h(self, h):   # of the next-h soonest tokens, how many land in q's top-h attended; per-step [T], use .nanmean()
    rel = (self.future_gap >= 1) & (future_gap <= h)        # [T, L]  soonest-h == relevant
    toph = self.S.topk(h, dim=-1).indices                       # [T, h]  predicted top-h candidates
    hit = rel.gather(-1, toph).double().sum(-1)                 # [T]
    recall = hit / h                                            # [T]

    valid = self.n_cand >= h                                    # need h relevant tokens to exist
    return self._nan_invalid(recall, valid)
# end


def pr_auc(self, h):   # average precision, positives = next-h soonest, scored by attention; per-step [T], use .nanmean()
    idx_sorted = self.S.argsort(dim=-1, descending=True)        # [T, L]  non-candidates (-inf) sink to the end
    gap_sorted = self.future_gap.gather(-1, idx_sorted)         # [T, L]

    is_cand = gap_sorted > 0                                    # [T, L]  a retrieved candidate at this rank
    is_pos = (gap_sorted >= 1) & (gap_sorted <= h)              # [T, L]  a soonest-h positive

    tp = is_pos.cumsum(-1).double()                            # [T, L]
    retrieved = is_cand.cumsum(-1).double().clamp_min(1.0)     # [T, L]  rank among candidates only
    precision = tp / retrieved                                # [T, L]

    P = is_pos.double().sum(-1)                                # [T]  == min(h, n_cand)
    ap = (precision * is_pos.double()).sum(-1) / P.clamp_min(1.0)   # [T]

    valid = self.n_cand > h                                    # need >=1 positive and >=1 negative
    return self._nan_invalid(ap, valid)
# end